# Reading a run

Everything downstream of a run reads one file: the web UI, every metric, and every
flow that rewinds a finished run.

This notebook generates its own run rather than pointing at one of yours, so it
works in a fresh clone before you have API keys. The run is real: real MCP server,
real game clock, real event log, with the model replaced by a script.

In [ ]:
import logging
import os
import tempfile
from pathlib import Path

from pytest import MonkeyPatch

# These notebooks generate their own run, so nothing here reaches a provider.
# Clearing the keys makes that a fact rather than a claim: if any cell below
# tried to call a model, it would fail here rather than spend.
for _key in ("ANTHROPIC_API_KEY", "OPENAI_API_KEY", "HF_TOKEN"):
    os.environ.pop(_key, None)

# The platform logs a line per tool call at INFO, which buries a notebook's own
# output in several hundred lines of it.
logging.getLogger("glossogen").setLevel(logging.WARNING)

SCENARIO = "warehouse_robot_recovery"
PRESET = "knobs_default"

In [ ]:
from glossogen.testing import run_rounds


async def generate_run(round_count, overrides):
    """Run the real round loop with the model replaced by a script.

    Every part of the platform is real here: the MCP server, the tool dispatch,
    the game clock, the world and the event logger. Only the LLM is scripted, so
    this costs nothing, needs no key, and gives the same answer every time.
    """
    with MonkeyPatch.context() as patch:
        return await run_rounds(
            scenario_name=SCENARIO,
            preset_name=PRESET,
            round_count=round_count,
            overrides=overrides,
            tmp_path=Path(tempfile.mkdtemp()),
            monkeypatch=patch,
        )


result = await generate_run(round_count=4, overrides={})
print(f"{len(result.events)} events at {result.log_path}")

## What a run is made of

One line per event, appended and never rewritten. That is what makes a byte offset
into this file a stable address, and it is why a finished run can be replayed from
any round.

In [ ]:
import collections

kinds = collections.Counter(event["event_type"] for event in result.events)
for kind, count in kinds.most_common():
    print(f"{count:5}  {kind}")

Most of a run is the agents thinking rather than speaking:
`llm_response_received` and `tool_call_invoked` dwarf `message_sent`. An agent
spends most of its turns reading notifications and calling tools.

The events that carry the scenario are the other two. `injection_delivered` is the
scenario telling an agent what happened this round; `world_event_delivered` is the
world answering what an agent did.

## Loading it the way the platform does

`load_events` parses each line into a typed event and backfills `round_number` on
older logs, so downstream code can read that field on any event.

In [ ]:
from glossogen.evaluation.log_reader import load_events

events = await load_events(log_path=result.log_path)
print(f"{len(events)} typed events")
print(f"rounds recorded: {len({e.round_number for e in events if e.round_number})}")

## One row per message

The primary channel is the one the throughput and language metrics read, and the
scenario declares it rather than the analysis guessing. Ask the scenario, so this
cell works unchanged on a scenario with two competing teams.

In [ ]:
import pandas as pd

primary = {channel.channel_id for channel in result.scenario.get_primary_channels()}
print(f"primary channel(s): {sorted(primary)}")

rows = [
    {
        "round": event["message"]["round_number"],
        "sender": event["message"]["sender_agent_id"],
        "channel": event["message"]["channel_id"],
        "chars": len(event["message"]["text"]),
        "text": event["message"]["text"],
    }
    for event in result.of_type(event_type="message_sent")
    if event["message"]["channel_id"] in primary
]
messages = pd.DataFrame(rows)
messages

## Characters per round

In this scenario a character is a cost: the agents spend a budget by talking.
`mean_chars_per_round` is this number, computed here by hand.

In [ ]:
import matplotlib.pyplot as plt

per_round = messages.groupby("round")["chars"].agg(["sum", "count"])
per_round.columns = ["chars", "messages"]
print(per_round)
print(f"\nmean chars/round: {per_round['chars'].mean():.1f}")

figure, axis = plt.subplots(figsize=(6, 3))
axis.bar(per_round.index, per_round["chars"], color="#0e6b5c")
axis.set_xlabel("round")
axis.set_ylabel("characters sent")
axis.set_title(f"{SCENARIO}: characters per round")
axis.set_xticks(list(per_round.index))
figure.tight_layout()

The scripted agents send the same short message every round, so this plot is
flat. Against a real run it is not, and the shape of it is the finding: agents
under budget pressure send less per round as they converge on a shorter code.

## Round verdicts

`judge_round_result` is a required hook, and the game clock writes what it returns
into `round_result_recorded`. That event is the only thing `round_success` reads,
which is why the metric works on any scenario without knowing anything about
robots.

In [ ]:
verdicts = result.of_type(event_type="round_result_recorded")
for verdict in verdicts:
    print(f"round {verdict['round_number']}: success={verdict['success']}")

passed = sum(1 for verdict in verdicts if verdict["success"])
print(f"\nround_success would report {passed}/{len(verdicts)}")

Next: [02_score_a_run.ipynb](02_score_a_run.ipynb) runs the real metric layer over
a run instead of computing numbers by hand.